In [7]:
import math

def viterbi(sentence, states, initial_pi, transition_a, emission_b):
    T = len(sentence)
    V = [{}]
    path = [{}]

    # Initialization [cite: 7]
    for state in states:
        # Use a small value (1e-10) to avoid log(0)
        prob = initial_pi.get(state, 0) * emission_b[state].get(sentence[0], 1e-10)
        V[0][state] = math.log(prob) if prob > 0 else -999
        path[0][state] = None

    # Recursion [cite: 5, 6]
    for t in range(1, T):
        V.append({})
        path.append({})
        for state in states:
            (prob, prev_state) = max(
                (V[t-1][prev_state] + 
                 math.log(transition_a[prev_state].get(state, 1e-10)) + 
                 math.log(emission_b[state].get(sentence[t], 1e-10)), prev_state)
                for prev_state in states
            )
            V[t][state] = prob
            path[t][state] = prev_state

    # Termination 
    (max_log_prob, last_state) = max((V[T-1][s], s) for s in states)

    # Backtracking
    opt_sequence = [last_state]
    for t in range(T-1, 0, -1):
        last_state = path[t][last_state]
        opt_sequence.insert(0, last_state)

    return opt_sequence, max_log_prob

# Data Definitions [cite: 12, 13, 18, 23]
STATES = ["Noun", "Verb", "Determiner"]
INITIAL_PI = {"Noun": 0.4, "Verb": 0.1, "Determiner": 0.5}
TRANSITION_A = {
    "Noun": {"Noun": 0.3, "Verb": 0.6, "Determiner": 0.1},
    "Verb": {"Noun": 0.5, "Verb": 0.2, "Determiner": 0.3},
    "Determiner": {"Noun": 0.8, "Verb": 0.1, "Determiner": 0.1}
}
EMISSION_B = {
    "Noun": {"the": 0.01, "chase": 0.40, "dogs": 0.40, "cats": 0.19},
    "Verb": {"the": 0.01, "chase": 0.70, "dogs": 0.15, "cats": 0.14},
    "Determiner": {"the": 0.98, "chase": 0.01, "dogs": 0.005, "cats": 0.005}
}

# --- User Input Handling ---
user_input = input("Enter a space-separated sentence: ").lower().split()
print(f"Input sentence: {' '.join(user_input)}")
result, log_score = viterbi(user_input, STATES, INITIAL_PI, TRANSITION_A, EMISSION_B)

print(f"\nOptimal Tag Sequence: {result}")
print(f"Final Log-score: {log_score:.4f}")

Input sentence: the chase

Optimal Tag Sequence: ['Determiner', 'Noun']
Final Log-score: -1.8528
